# 실제 촬영 데이터 검증: 시뮬레이션으로 학습한 side plate 모델 (GT 있어도, 없어도)

시뮬레이션(Isaac Sim)으로만 학습한 파이프라인 **Mask R-CNN detector → GDRN 방식 RGB-D pose 네트워크 → depth anchor → ICP → GT 없는
confidence 판정**을 실제 Photoneo 촬영에 적용하고, 6D pose가 잘 추정되는지 판단합니다. 데이터는 두 종류를 받습니다.

- **GT 없는 새 촬영** (`camera_convert_data.py`로 찍은 `camera/`): BOP 형식으로 자동 변환하고(렌즈 왜곡 보정 포함) 추론합니다.
  정답이 없으므로 **GT 없이 잴 수 있는 기준**(ICP 맞춤 오차, CAD 표면과 실측 depth의 모순, 같은 장면 반복 촬영의 흔들림 등)으로 판단합니다.
  기준값은 의사 정답이 있는 실제 촬영(`data/pseudo_gt_20_bop`)에서 **같은 기준으로 잰 값**에 맞춰 정했습니다(§3-2).
- **의사 정답(GT)이 있는 세트** (`data/pseudo_gt_20_bop` 등): 위의 GT 없는 기준에 더해 의사 정답 대비 정확도(ADD-S, 위치, 기울기)가 나옵니다.

**촬영 방법** (harvesters venv에서 실행, 학습과 같은 카메라 설정 1236×1032으로 찍힘):
```bash
~/Projects/photoneo-python-examples/GigE-V/harvesters/.venv/bin/python camera_convert_data.py --count 3      # 같은 장면 3장
~/Projects/photoneo-python-examples/GigE-V/harvesters/.venv/bin/python camera_convert_data.py --interactive  # Enter마다 1장
```
배치마다 **움직이지 말고 2~3장 연속으로** 찍으면 반복성(§4-2)을 볼 수 있습니다. 배치를 바꿔 가며 10장 이상 찍고, 장마다 놓은 부품 수를 알면
아래 `PARTS_PER_FRAME`에 적어 주세요(찾은 비율이 나옵니다).

**사용한 모델:** `runs/detector/best.pt`, `runs/pose/best.pt`. 둘 다 side plate 시뮬레이션 데이터로 학습한 모델입니다.
`pretrained/gdrnpp_itodd_2.pth`는 side plate 모델이 **아닙니다**. GDRNPP 공식 ITODD 체크포인트이고, pose 학습을
시작할 때 backbone 초기값으로만 쓰입니다.

**정답에 대한 주의:** `data/pseudo_gt_20_bop`의 정답은 FoundationPose가 추정한 **의사 정답**입니다. 오차는 진짜 오차가 아니라
**FoundationPose 결과와의 차이**이고, 의사 정답 자체에 카메라 방향 위치 약 0.5mm, 기울기 1~2° 수준의 불확실성이 있습니다
(`data/pseudo_gt_20_bop/README.txt`).

판정은 §6에 표로 나옵니다. 모든 숫자는 추론 결과(`runs/...`)에서 매번 다시 계산합니다.

In [ ]:
# ---- 설정 -------------------------------------------------------------------------------------------------
CAPTURE_DIR = "camera"                 # camera_convert_data.py 촬영 폴더. None = REAL_ROOT를 그대로 사용 (변환 안 함)
REAL_ROOT = "data/real_capture_bop"    # 변환된 BOP 데이터셋 위치. 의사 정답 세트: "data/pseudo_gt_20_bop" + CAPTURE_DIR = None
RUN_PREFIX = "runs/real_capture"       # 추론 결과 위치 (<prefix>_full, <prefix>_gtmask). 의사 정답 세트: "runs/photoneo"
UNDISTORT = True                       # 렌즈 왜곡 보정 (촬영 폴더의 undistort_map.npy 필요)
PARTS_PER_FRAME = None                 # 장마다 실제로 놓은 부품 수 (예: 5). 모르면 None
REF_ROOT, REF_RUN = "data/pseudo_gt_20_bop", "runs/photoneo_full"   # 기준: 의사 정답이 있는 실제 촬영과 그 추론 결과

# GT 없는 판정 기준값: 기준 run에서 의사 정답과 1mm 이내인 confident pose의 분포로 정함 (§3-2 보정 표)
CRIT = {"fit_rmse_mm": 0.7,       # confident 맞춤 오차 중앙값 상한 (기준: 0.47mm, 1mm 단위 depth)
        "fitness": 0.98,          # confident fitness 중앙값 하한 (기준: 0.999)
        "violation": 0.10,        # 이 비율 넘게 depth가 CAD 표면 '뒤'에 있으면 모순 (기준 confident: p95 0.06, 틀린 pose: >= 0.28)
        "violation_share": 0.05,  # 모순인 confident의 허용 비율
        "repeat_centre_mm": 0.5,  # 같은 장면 반복 촬영에서 위치 흔들림 상한
        "repeat_tilt_deg": 1.0,   # 같은 장면 반복 촬영에서 기울기 흔들림 상한
        "cam_f_rel": 0.01,        # 학습 카메라와 초점거리 1% 이내 (주점은 K로 처리되므로 보지 않음)
        "dist_rel": 0.15}         # 카메라-부품 거리가 학습(시뮬레이션) 중앙값에서 15% 이내

import os, subprocess, sys
from pathlib import Path
if Path.cwd().name == "notebooks":                      # run from the project root
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import json
import numpy as np, pandas as pd, plotly.graph_objects as go
from IPython.display import display
from PIL import Image

from data_conversion.bin_dataset import BinPoseDataset
from bin6d.config import Cfg, load_config
from bin6d.pose_data import ObjectModel
from bin6d.report import (accuracy_table, detections_table, frame_index, gt_free_table, load_frame, load_run,
                          per_frame_table, repeatability_table, static_groups)
from bin6d.viz import gt_overlay_sets, plotly_scene, render_2d, select
from data_conversion.camera_convert_data import export_bop

cfg = load_config("configs/default.yaml")
pd.set_option("display.width", 200)

# ---- 촬영 폴더 -> BOP 변환 (촬영이 바뀌었을 때만) ----
if CAPTURE_DIR:
    cap = Path(CAPTURE_DIR)
    cam_json = json.load(open(cap / "camera.json"))
    undistort = UNDISTORT and "undistort_map" in cam_json
    if UNDISTORT and not undistort:
        print("주의: camera.json에 undistort_map이 없습니다(이전 버전 촬영 스크립트). 왜곡 보정 없이 변환합니다. "
              "camera_convert_data.py로 다시 찍으세요.")
    sig = {"captures": sorted((p.name, p.stat().st_mtime_ns) for p in (cap / "rgb").glob("*.png")),
           "camera": cam_json, "undistort": undistort}
    stamp = Path(REAL_ROOT) / "export_signature.json"
    if not stamp.exists() or json.load(open(stamp)) != json.loads(json.dumps(sig)):
        kept = export_bop(cap, REAL_ROOT, undistorted=undistort)
        stamp.write_text(json.dumps(sig))
        print(f"{CAPTURE_DIR} -> {REAL_ROOT}: {len(kept)} frames (undistorted: {undistort})")

real = BinPoseDataset(REAL_ROOT)
SCENE = real.frames[0].scene
ridx = frame_index(real)
model = ObjectModel(real, cfg.data.obj_id, 0, 3000)
HAS_GT = any(len(real.frame_meta(i)["objects_all"]["t"]) for i in range(len(real)))
manifest = json.load(open(Path(REAL_ROOT) / "manifest.json")) if (Path(REAL_ROOT) / "manifest.json").exists() else None
sources = json.load(open(Path(REAL_ROOT) / "sources.json")) if (Path(REAL_ROOT) / "sources.json").exists() else {}
UNDISTORTED = sources.get("0", {}).get("undistorted")          # None = unknown (dataset not made by export_bop)

DET, GTM = "detector masks (real pipeline)", "ground-truth masks"
RUNS = {DET: f"{RUN_PREFIX}_full"} | ({GTM: f"{RUN_PREFIX}_gtmask"} if HAS_GT else {})
SIM_RUNS = {DET: "runs/eval_full", GTM: "runs/eval_gtmask"}

# reference: real capture with pseudo-GT (skipped if it is the data under test or not on this machine)
ref = None
if Path(REF_ROOT).exists() and list(Path(REF_RUN).glob("*_*.json")) and Path(REF_ROOT).resolve() != Path(REAL_ROOT).resolve():
    ref = BinPoseDataset(REF_ROOT)

# simulation held-out scene, for side-by-side numbers (skipped if the sim data is not on this machine)
try:
    VAL = cfg.data.val_scenes[0]
    sim = BinPoseDataset(cfg.data.root, scenes=[VAL])
    sidx = frame_index(sim)
except (FileNotFoundError, ValueError) as e:
    sim = None
    print(f"simulation data not available ({e}); sim comparisons are skipped")

S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
def style(fig, title, height=360):
    fig.update_layout(template="plotly_white", title=title, height=height, font=dict(color="#52514e"),
                      margin=dict(l=60, r=20, t=60, b=40), legend=dict(orientation="h", y=1.12, x=0))
    fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    return fig
print(f"data: {REAL_ROOT}, {len(real)} frames ({real.split}), GT: {'yes' if HAS_GT else 'no'}, "
      f"part diameter {model.diameter:.1f} mm"
      + (f" | reference: {REF_ROOT}" if ref else "") + (f" | sim val scene {VAL}: {len(sim)} frames" if sim else ""))

## 1. 데이터

프레임마다 depth가 있는 픽셀 비율, depth 단위(양자화 간격), 밝기를 봅니다. **same scene as**는 depth가 거의 그대로인 연속 프레임
(부품을 움직이지 않고 다시 찍은 것)을 묶은 것이고, §4-2 반복성에 쓰입니다.

**카메라 설정 확인:** 네트워크와 detector는 1236×1032, 초점거리 약 1412px 이미지로만 학습했습니다. 해상도나 초점거리가 다르면
부품이 다른 크기로 보이고, 네트워크는 깊이를 겉보기 크기로 추정하므로 결과를 믿을 수 없습니다. 아래 표에서 **OK**가 아니면 다시 찍으세요.

In [ ]:
def depth_step(d):
    vals = np.unique(d[d > 0])
    return float(np.median(np.diff(vals))) if len(vals) > 1 else float("nan")

groups = static_groups(real)
group_of = {v: (f"{g[0]}-{g[-1]}" if len(g) > 1 else "-") for g in groups for v in g}
rows = []
for i, fr in enumerate(real.frames):
    f = load_frame(real, i)
    rows.append({"view": fr.view, "capture": sources.get(str(fr.view), {}).get("capture", "-"),
                 "same scene as": group_of[fr.view], "pixels with depth": f"{f['valid'].mean():.0%}",
                 "scene depth median (mm)": round(float(np.median(f["depth"][f["valid"]])), 1),
                 "depth step (mm)": round(depth_step(f["depth"]), 2), "brightness (0-255)": round(float(f["rgb"].mean()), 1),
                 "GT parts": len(f["objects_all"]["t"])})
frames_df = pd.DataFrame(rows).set_index("view")
display(frames_df.T if len(frames_df) <= 20 else frames_df)

# camera vs the camera the models were trained with
def camera_of(frames):
    m = frames.frame_meta(0)
    w, h = Image.open(frames.frames[0].shard / frames.split / frames.frames[0].scene / "rgb" / f"{frames.frames[0].view:06d}.png").size
    K = m["K"]
    return {"width": w, "height": h, "fx": float(K[0, 0]), "fy": float(K[1, 1]), "cx": float(K[0, 2]), "cy": float(K[1, 2])}

sim_cam = sorted(Path(cfg.data.root).glob("shard_*/camera.json"))
TRAIN_CAM = (json.load(open(sim_cam[0])) if sim_cam else camera_of(ref or real))
this_cam = camera_of(real)
# resolution and focal length set the apparent part size the networks learned; the principal point is handled by K
CAM_OK = ((this_cam["width"], this_cam["height"]) == (TRAIN_CAM["width"], TRAIN_CAM["height"])
          and all(abs(this_cam[k] / TRAIN_CAM[k] - 1) <= CRIT["cam_f_rel"] for k in ("fx", "fy")))
cams = {"this data": this_cam, "training (sim)": {k: TRAIN_CAM[k] for k in this_cam}}
if ref:
    cams["reference (pseudo-GT real)"] = camera_of(ref)
cam_df = pd.DataFrame(cams).T.round(2)
cam_df["lens distortion corrected"] = [{True: "yes", False: "NO", None: "unknown"}[UNDISTORTED], "n/a (pinhole)", "unknown"][:len(cam_df)]
display(cam_df)
print(f"camera setup vs training: {'OK' if CAM_OK else 'DIFFERENT -> recapture with camera_convert_data.py (1236x1032, PrimaryCamera)'}")

In [ ]:
def strip(f, depth_range=None, h=300):
    lo, hi = depth_range or np.percentile(f["depth"][f["valid"]], [2, 98])
    dv = np.clip((f["depth"] - lo) / max(hi - lo, 1), 0, 1)
    dv = (np.where(f["valid"], 1 - dv, 0) * 255).astype(np.uint8)
    tiles = [Image.fromarray(f["rgb"]), Image.fromarray(dv).convert("RGB")]
    if f["inst"].max() > 0:
        rng = np.random.default_rng(0); lut = rng.integers(40, 255, (f["inst"].max() + 1, 3)); lut[0] = 0
        tiles.append(Image.fromarray(lut[f["inst"]].astype(np.uint8)))
    w = int(h * f["rgb"].shape[1] / f["rgb"].shape[0])
    out = Image.new("RGB", (len(tiles) * w, h))
    for i, t in enumerate(tiles):
        out.paste(t.resize((w, h)), (w * i, 0))
    return out
for v in sorted({real.frames[0].view, real.frames[len(real) // 2].view}):
    print(f"view {v}: RGB | depth (black = no depth)" + (" | pseudo-GT visible masks" if HAS_GT else ""))
    display(strip(load_frame(real, ridx[(SCENE, v)])))

### 시뮬레이션, 기준 실제 촬영과의 차이

장면 전체 기준 값은 GT 없이도 나옵니다. 부품 픽셀 기준 값(거리, 크기, 밝기)은 GT 마스크가 있는 데이터에서만 나옵니다
(GT 없는 데이터의 카메라-부품 거리는 §3-2에서 confident 검출로 잽니다).

In [ ]:
def frame_stats(frames, i):
    f = load_frame(frames, i)
    s = {"scene depth median (mm)": float(np.median(f["depth"][f["valid"]])),
         "pixels with depth": float(f["valid"].mean()), "image brightness (0-255)": float(f["rgb"].mean()),
         "depth step (mm)": depth_step(f["depth"])}
    o, part = f["objects_all"], f["inst"] > 0
    if len(o["t"]) and part.any():
        s.update({"camera -> part distance (mm)": float(np.median(o["t"][:, 2])),
                  "visible part size (px, median)": float(np.median(o["visib_px"][o["visible"]])),
                  "part pixels with depth": float((part & f["valid"]).sum() / part.sum()),
                  "part brightness (0-255)": float(f["rgb"][part].mean()), "parts per frame": len(o["t"])})
    return s

cols = {f"this data ({len(real)} frames)": pd.DataFrame([frame_stats(real, i) for i in range(len(real))]).median()}
if ref:
    cols[f"reference real ({len(ref)} frames)"] = pd.DataFrame([frame_stats(ref, i) for i in range(len(ref))]).median()
if sim:
    cols[f"sim (scene {VAL}, 10 frames)"] = pd.DataFrame([frame_stats(sim, sidx[(VAL, v)]) for v in range(0, 100, 10)]).median()
display(pd.DataFrame(cols).round(3))
TRAIN_DIST = float(cols[f"sim (scene {VAL}, 10 frames)"]["camera -> part distance (mm)"]) if sim else 700.0

## 2. 추론 실행

`infer.py`로 **detector 마스크**(실제 파이프라인)를 돌립니다. GT가 있으면 detector를 건너뛰고 pose 품질만 보는 **GT 마스크** 설정도 돌립니다.
결과(JSON과 오버레이 PNG)가 데이터와 맞으면 다시 실행하지 않습니다. 촬영이 바뀌면 자동으로 다시 돌고, 모델을 바꿨다면 `RERUN = True`로 바꾸세요.

```bash
.venv/bin/python infer.py --set data.root=<REAL_ROOT> --scenes 000000 --view-stride 1 --out <RUN_PREFIX>_full
.venv/bin/python infer.py --set data.root=<REAL_ROOT> --scenes 000000 --view-stride 1 --out <RUN_PREFIX>_gtmask --gt-masks   # GT 있을 때
```

In [ ]:
RERUN = False
want = {f"{fr.scene}_{fr.view:06d}" for fr in real.frames}
stamp = Path(REAL_ROOT) / "export_signature.json"
for name, out in RUNS.items():
    done = Path(out) / "summary.json"                    # written last by infer.py (frames without detections get no JSON)
    extra = {p.stem for p in Path(out).glob("*_*.json")} - want
    stale = done.exists() and stamp.exists() and done.stat().st_mtime < stamp.stat().st_mtime
    if RERUN or not done.exists() or stale or extra:
        for p in Path(out).glob("*_*.*"):                # old per-frame outputs (views are renumbered on export)
            p.unlink()
        cmd = [sys.executable, "infer.py", "--set", f"data.root={REAL_ROOT}", "--scenes", SCENE,
               "--view-stride", "1", "--out", out] + (["--gt-masks"] if name == GTM else [])
        print(" ".join(cmd))
        subprocess.run(cmd, check=True, env={**os.environ, "HF_HUB_OFFLINE": "1"})
    print(f"{name}: {len(list(Path(out).glob('*_*.json')))} frames in {out}")
runs = {name: load_run(path, cfg.refine) for name, path in RUNS.items()}

## 3. 정확도

### 3-1. 의사 정답 대비 (GT가 있을 때만)

**ADD-S**는 예측 pose와 정답 pose에서 CAD 표면 점들 사이의 평균 거리입니다(대칭 중 최솟값). **centre err**는 위치 오차, **tilt**는 판 법선 사이의 각도입니다.
`n`은 정답과 매칭된 검출 수입니다(마스크 IoU ≥ 0.5). 5.6mm는 부품 지름의 10%로, BOP 표준 기준입니다.

In [ ]:
tables = {}
if HAS_GT:
    tables = {name: detections_table(runs[name], real, model) for name in RUNS}
    for name, df in tables.items():
        print(f"\n{name}: {len(df)} detections, {df.matched.sum()} matched to pseudo-GT (of {sum(len(real.frame_meta(i)['objects_all']['t']) for i in range(len(real)))} parts)")
        display(accuracy_table(df, diameter=model.diameter))
else:
    print("GT 없음: 3-1은 건너뜁니다. 3-2의 GT 없는 기준을 보세요.")

같은 모델을 시뮬레이션 검증 장면(`runs/eval_*`, 20장)에 적용한 결과와 나란히 비교합니다(GT가 있을 때). 시뮬레이션은 정확한 GT, 실제는 의사 정답입니다.

**읽는 법:** confident인 pose가 의사 정답과 1mm 이내인 비율(**confident: ADD-S < 1 mm**)이 핵심입니다. 이 값이 높으면 로봇이 confident만 집어도 됩니다.
**confident (share of matched)**가 낮으면 쓸 수 있는 pose가 적다는 뜻이고, 원인은 §4에서 봅니다.

In [ ]:
def summary(df):
    m = df[df.matched]
    c = m[m.confident]
    return {"matched detections": len(m),
            "network only: ADD-S < 5.6 mm": f"{(m.net_add_s < 0.1 * model.diameter).mean():.0%}",
            "network only: centre err median (mm)": round(m.net_centre_mm.median(), 1),
            "ICP accepted": f"{m.icp_refined.mean():.0%}",
            "confident (share of matched)": f"{m.confident.mean():.0%}",
            "confident: ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}",
            "confident: centre err median (mm)": round(c.final_centre_mm.median(), 2),
            "confident: tilt median (deg)": round(c.final_tilt_deg.median(), 2)}

if HAS_GT:
    cols = {}
    for name in RUNS:
        short = "detector" if name == DET else "GT masks"
        if sim and list(Path(SIM_RUNS[name]).glob("*_*.json")):
            cols[f"sim, {short}"] = summary(detections_table(load_run(SIM_RUNS[name], cfg.refine), sim, model))
        cols[f"real, {short}"] = summary(tables[name])
    display(pd.DataFrame(cols))

    # detector quality: where are the false detections? near a real part (duplicate / partial mask) or on empty bin
    df = tables[DET]
    gt_parts = sum(len(real.frame_meta(i)["objects_all"]["t"]) for i in range(len(real)))
    fp = df[~df.matched]
    display(pd.DataFrame({
        "pseudo-GT parts": [gt_parts],
        "found (IoU >= 0.5)": [f"{df.matched.sum()} ({df.matched.sum() / gt_parts:.0%})"],
        "false detections": [len(fp)],
        "false detections per frame": [round(len(fp) / df.view.nunique(), 1)],
        "false detections flagged confident": [int(fp.confident.sum())],
        "false detections: median depth points": [int(fp.n_depth_points.median()) if len(fp) else 0],
    }, index=["detector"]).T)
    rows = []
    for (s, v), dets in runs[DET].items():
        o = real.frame_meta(ridx[(s, v)])["objects_all"]
        for d in dets:
            if d["gt_inst_id"] is None:
                dist = float(np.min(np.linalg.norm(o["t"][:, :2] - np.asarray(d["t_rgbd"])[:2], axis=1)))
                rows.append({"kind": "on a real part (duplicate / partial mask)" if dist < 40 else "no part nearby (bin rim / background)",
                             "score": d["score"], "depth points": d["n_depth_points"]})
    if rows:
        g = pd.DataFrame(rows).groupby("kind")
        display(pd.DataFrame({"false detections": g.size(), "score median": g.score.median().round(2),
                              "depth points median": g["depth points"].median()}))

### 3-2. GT 없는 기준

정답이 없어도 **pose와 실측 depth가 서로 맞는지**는 잴 수 있습니다. 검출마다 다음을 계산합니다.

| 신호 | 뜻 | 좋은 pose |
|---|---|---|
| **ICP fitness / fit error** | 마스크 안 실측 점이 CAD 표면에 얼마나 붙는지 (confident 판정에 이미 쓰임) | fitness ≈ 1, fit error ≈ depth 노이즈 |
| **depth 모순 (violation)** | 그 pose의 CAD 보이는 면을 투영했을 때, 실측 depth가 CAD 표면보다 **3mm 넘게 뒤에** 있는 비율. 센서가 부품이 있어야 할 곳을 뚫고 본다는 뜻이라, pose가 그 자리에 있을 수 없습니다. 가려짐(앞에 다른 물체)은 모순으로 세지 않습니다. ICP와 달리 마스크 밖까지 보므로 마스크 일부에만 맞춘 pose를 잡아냅니다 | ≈ 0 |
| **surface support** | 투영한 CAD 면 중 실측 depth가 3mm 안에 있는 비율 | 높음 (가려지면 낮아짐) |
| **anchor dz / ICP 보정량** | 네트워크 pose를 실측 depth와 ICP가 얼마나 옮겼는지. 네트워크 자체의 오차를 GT 없이 보여 줍니다 | 작음 |

아래 **보정 표**는 의사 정답이 있는 실제 촬영(기준 run)에서 이 신호들이 맞는 pose와 틀린 pose를 실제로 가르는지 보여 줍니다.
§6의 판정 기준값(`CRIT`)은 이 표에서 정했습니다.

In [ ]:
def classes(g):
    return np.select([g.confident & g.matched & (g.final_add_s < 1), g.confident & g.matched, g.matched],
                     ["confident, < 1 mm", "confident, >= 1 mm", "not confident (real part)"], "false detection (no part)")

SIG = ["icp_fitness", "icp_rmse", "surf_violation", "surf_support", "anchor_dz_mm", "icp_shift_mm", "icp_rot_deg",
       "n_depth_points"]
free = gt_free_table(runs[DET], real, model)
ref_free = gt_free_table(load_run(REF_RUN, cfg.refine), ref, model) if ref else None
calib = ref_free if ref_free is not None else (free if HAS_GT else None)
if calib is not None:
    calib = calib.assign(cls=classes(calib))
    g = calib.groupby("cls")
    t = g[SIG].median().round(3)
    t.insert(0, "detections", g.size())
    t["share violation > %.2f" % CRIT["violation"]] = g.surf_violation.apply(lambda s: f"{(s > CRIT['violation']).mean():.0%}")
    print(f"calibration on {'reference ' + REF_RUN if ref_free is not None else 'this data'} (medians, pseudo-GT labels)")
    display(t.T)

**이 데이터의 GT 없는 요약.** 기준 열은 같은 모델을 의사 정답이 있는 실제 촬영에 돌린 값입니다(그 confident pose는 의사 정답과 거의 모두 1mm 이내).
이 데이터의 값이 기준과 비슷하면, 같은 수준으로 맞는다고 볼 근거가 됩니다.

In [ ]:
def free_summary(g, n_frames, placed=None):
    c, acc = g[g.confident], g[g.icp_refined]
    nc = g[~g.confident]
    out = {"frames": n_frames, "detections per frame": round(len(g) / n_frames, 1),
           "ICP accepted": f"{g.icp_refined.mean():.0%}" if len(g) else "-",
           "confident per frame": round(len(c) / n_frames, 1),
           "confident: fit error median (mm)": round(c.icp_rmse.median(), 3) if len(c) else np.nan,
           "confident: fitness median": round(c.icp_fitness.median(), 3) if len(c) else np.nan,
           "confident: surface support median": round(c.surf_support.median(), 2) if len(c) else np.nan,
           f"confident with depth contradiction (> {CRIT['violation']:.0%})": int((c.surf_violation > CRIT["violation"]).sum()),
           "confident: camera -> part distance median (mm)": round(c.distance_mm.median(), 1) if len(c) else np.nan,
           "all ICP-accepted: fit error median (mm)": round(acc.icp_rmse.median(), 3) if len(acc) else np.nan,
           "not confident: too few depth points": int((nc.n_depth_points < cfg.refine.min_points).sum()),
           f"not confident: depth contradiction (> {CRIT['violation']:.0%})": int((nc.surf_violation > CRIT["violation"]).sum())}
    if placed:
        out["confident / placed parts"] = f"{len(c) / (placed * n_frames):.0%}"
    return out

cols = {"this data": free_summary(free, len(real), PARTS_PER_FRAME)}
if ref_free is not None:
    cols["reference real (pseudo-GT)"] = free_summary(ref_free, len(ref))
if sim and list(Path(SIM_RUNS[DET]).glob("*_*.json")):
    sim_free = gt_free_table(load_run(SIM_RUNS[DET], cfg.refine), sim, model)
    cols["sim (held-out scene)"] = free_summary(sim_free, sim_free.view.nunique())
display(pd.DataFrame(cols))

**읽는 법:**
- **confident per frame**는 쓸 수 있는 pose 수입니다. 장마다 놓은 부품 수(`PARTS_PER_FRAME`)를 알면 **confident / placed parts**로 비율이 나옵니다.
- **confident fit error**가 기준과 비슷하고(실제 depth 노이즈 수준), **depth contradiction**이 0에 가까우면 confident pose가 실측 depth와 모순 없이 맞는 것입니다.
- **not confident** 중 depth 점이 거의 없거나 depth 모순이 큰 검출은 대부분 오검출(부품이 없는 곳, 중복 마스크)입니다. confident로 표시되지 않으므로 로봇이 confident만 집는다면 문제가 되지 않습니다.

## 4. 실패 원인: 네트워크의 깊이(z) 오차

ICP는 보정량이 회전 30°나 이동 15mm를 넘으면 결과를 거부합니다. confident가 적다면 대부분 네트워크 pose가 ICP 범위 밖이기 때문입니다.
GT가 있으면 네트워크 오차를 정답과 비교해 방향별로 나누고, GT가 없으면 **depth anchor가 옮긴 거리(anchor dz)**와 **ICP 보정량**으로 봅니다
(anchor는 네트워크 pose를 카메라 광선을 따라 실측 depth까지 옮기므로, anchor dz ≈ -(네트워크 깊이 오차)).

In [ ]:
def net_error_parts(run, frames):
    idx = frame_index(frames)
    rows = []
    for (s, v), dets in run.items():
        o = frames.frame_meta(idx[(s, v)])["objects_all"]
        g = {int(i): k for k, i in enumerate(o["inst_id"])}
        for d in dets:
            if d["gt_inst_id"] is None:
                continue
            k = g[d["gt_inst_id"]]
            dt = np.asarray(d["t_rgbd"]) - o["t"][k]
            Rrel = o["R"][k].T @ np.asarray(d["R_rgbd"])
            a = np.degrees(np.arctan2(Rrel[1, 0], Rrel[0, 0])) % 180     # in-plane angle, 2-fold symmetric
            rows.append({"view": v, "ICP accepted": d["icp_refined"], "dz (mm, + = too far)": dt[2],
                         "|dxy| (mm)": float(np.linalg.norm(dt[:2])), "in-plane rot (deg)": min(a, 180 - a)})
    return pd.DataFrame(rows)

if HAS_GT:
    errs = {name: net_error_parts(runs[name], real) for name in RUNS}
    for name, e in errs.items():
        g = e.groupby("ICP accepted")
        print(f"\n{name}: network error before anchor/ICP vs pseudo-GT (median), split by ICP outcome")
        display(pd.DataFrame({"detections": g.size(), "dz median (mm)": g["dz (mm, + = too far)"].median().round(1),
                              "share |dz| > 15 mm": g["dz (mm, + = too far)"].apply(lambda s: f"{(s.abs() > 15).mean():.0%}"),
                              "|dxy| median (mm)": g["|dxy| (mm)"].median().round(1),
                              "in-plane rot median (deg)": g["in-plane rot (deg)"].median().round(1)}))

# label-free: how far depth anchoring and ICP moved the network pose (detections with enough depth points)
def move_table(g):
    g = g[g.n_depth_points >= cfg.refine.min_points]
    k = g.groupby("icp_refined")
    return pd.DataFrame({"detections": k.size(), "anchor dz median (mm)": k.anchor_dz_mm.median().round(1),
                         "share |anchor dz| > 15 mm": k.anchor_dz_mm.apply(lambda s: f"{(s.abs() > 15).mean():.0%}"),
                         "ICP shift median (mm)": k.icp_shift_mm.median().round(2),
                         "ICP rotation median (deg)": k.icp_rot_deg.median().round(2),
                         "depth contradiction median": k.surf_violation.median().round(2)}).rename(
        index={True: "ICP accepted", False: "ICP not accepted"})
print("this data: pose changes after the network, split by ICP outcome (GT-free)")
display(move_table(free))
if ref_free is not None:
    print("reference real (pseudo-GT)")
    display(move_table(ref_free))

e = free[free.n_depth_points >= cfg.refine.min_points]
fig = go.Figure()
for ok, c, label in ((True, S3, "ICP accepted"), (False, S2, "ICP not accepted")):
    fig.add_trace(go.Histogram(x=-e[e.icp_refined == ok].anchor_dz_mm, name=label, marker_color=c,
                               xbins=dict(start=-40, end=60, size=2), opacity=0.85))
fig.add_vline(x=15, line_dash="dash", line_color="#52514e", annotation_text="ICP limit 15 mm")
style(fig, "Network depth error estimated from the depth anchor (this data, GT-free)").update_layout(barmode="stack")
fig.update_xaxes(title="-anchor dz = network z - measured surface z (mm), + = too far").update_yaxes(title="detections")
fig.show()

**읽는 법:** ICP가 받아들인 검출은 anchor 뒤 ICP 보정량이 작아야 합니다(수 mm 이하). ICP가 거부한 검출이 많고 그 anchor dz가 크면,
네트워크 깊이가 실측과 많이 어긋난 것입니다(시뮬레이션과 실제의 차이). anchor dz가 작은데도 거부됐다면 회전이나 마스크(오검출) 문제입니다.

### confidence 기준 비교

같은 추론 결과에 confidence 기준만 바꿔 다시 판정합니다. GT가 있으면 confident 중 1mm 이내 비율도 나옵니다.

In [ ]:
def variant(path, frames, min_fit, ratio, cap):
    rc = Cfg(dict(cfg.refine)); rc.confidence_min_fitness, rc.confidence_rmse_ratio, rc.confidence_max_rmse_mm = min_fit, ratio, cap
    d = detections_table(load_run(path, rc), frames, model)
    out = {"confident": int(d.confident.sum()), "per frame": round(d.confident.sum() / d.view.nunique(), 1)}
    if HAS_GT:
        c = d[d.confident & d.matched]
        out.update({"confident but unmatched": int((d.confident & ~d.matched).sum()),
                    "ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}" if len(c) else "-",
                    "wrong (>= 1 mm)": int((c.final_add_s >= 1).sum())})
    return out
for name, path in RUNS.items():
    print(name)
    display(pd.DataFrame({"fixed 0.8 / 1.0 mm": variant(path, real, 0.8, 1e9, 1.0), "fixed 0.9 / 0.5 mm": variant(path, real, 0.9, 1e9, 0.5),
                          "relative 0.9 / 1.3x median (cap 1.5 mm) [current]": variant(path, real, 0.9, 1.3, 1.5)}).T)
acc = free[free.icp_refined]
if len(acc):
    print(f"ICP-accepted fits on this data: fitness median {acc.icp_fitness.median():.3f}, "
          f"fit error (RMSE) median {acc.icp_rmse.median():.2f} mm")

고정 기준(0.5mm)은 depth 노이즈가 조금만 커져도 맞는 pose를 통째로 잃습니다. 프레임마다 기준을 맞추는 상대 기준(현재 설정)이 촬영 조건 변화에 안전합니다.

### 프레임별 결과

In [ ]:
k = free.groupby("view")
pf = pd.DataFrame({"same scene as": pd.Series(group_of), "detections": k.size(), "ICP accepted": k.icp_refined.sum(),
                   "confident": k.confident.sum(),
                   "median fit error (mm)": free[free.icp_refined].groupby("view").icp_rmse.median().round(3),
                   "confident: max depth contradiction": free[free.confident].groupby("view").surf_violation.max().round(3)})
pf.index.name = "view"
pf[["detections", "ICP accepted", "confident"]] = pf[["detections", "ICP accepted", "confident"]].fillna(0).astype(int)
if PARTS_PER_FRAME:
    pf["confident / placed"] = (pf.confident / PARTS_PER_FRAME).map("{:.0%}".format)
display(pf.T if len(pf) <= 20 else pf)

if HAS_GT:
    for name in RUNS:
        print(name); display(per_frame_table(tables[name], real).T)
        if manifest:                                     # pseudo_gt_20_bop: views 13-19 have touching / leaning parts
            m = tables[name][tables[name].matched]
            g = m.groupby(np.where(m.view >= 13, "overlapping (13-19)", "separated (0-12)"))
            display(pd.DataFrame({"matched": g.size(), "confident": g.confident.mean().map("{:.0%}".format),
                                  "confident ADD-S < 1 mm": g.apply(lambda s: f"{(s[s.confident].final_add_s < 1).mean():.0%}")}))

### 4-2. 반복성: 같은 장면을 여러 번 찍었을 때

부품을 움직이지 않고 연속으로 찍은 프레임(§1의 **same scene as**)에서 같은 부품의 confident pose가 얼마나 흔들리는지 봅니다.
GT 없이 잴 수 있는 **정밀도**입니다(치우침, 즉 정확도는 알 수 없음). 흔들림이 의사 정답의 불확실성(위치 약 0.5mm, 기울기 1~2°)보다
충분히 작아야 합니다. **seen in**이 그룹 장수보다 작으면 그 부품은 어떤 장에서는 confident가 아니었다는 뜻입니다(판정이 불안정한 부품).

In [ ]:
rep = repeatability_table(runs[DET], SCENE, groups)
if len(rep):
    display(rep.round(2))
    print(f"centre spread max: median {rep['centre spread max (mm)'].median():.2f} mm, worst {rep['centre spread max (mm)'].max():.2f} mm | "
          f"tilt max: median {rep['tilt max (deg)'].median():.2f} deg | "
          f"in-plane rot max: median {rep['in-plane rot max (deg)'].median():.2f} deg")
    cnt = pd.DataFrame([{"group": f"{g[0]}-{g[-1]}", "confident per frame": [int(pf.confident.get(v, 0)) for v in g]}
                        for g in groups if len(g) >= 2])
    display(cnt.set_index("group"))
else:
    print("같은 장면을 연속으로 찍은 프레임(또는 그 안의 confident 검출)이 없어 건너뜁니다. "
          "배치마다 camera_convert_data.py --count 3 으로 찍으면 나옵니다.")

## 5. 시각 확인

2D 오버레이: 각 검출의 3D 박스를 투영해 그렸습니다. GT가 없으면 **초록** = confident, **주황** = confident 아님입니다.
GT가 있으면 **초록** = confident이고 의사 정답과 1mm 이내, **빨강** = confident인데 틀림, **주황** = confident 아님,
**얇은 흰색** = pose가 맞지 않은 곳의 의사 정답, **자홍** = detector가 놓친 부품입니다. confident인 부품에는 xyz 축을 그립니다(x 빨강, y 초록, z 파랑).
아래는 첫 장, confident가 가장 적은 장, 검출이 가장 많은 장이고, 마지막은 첫 장의 confident만 그린 것입니다(부품이 많은 bin에서 보기 쉬움).

In [ ]:
def overlay(view, show="all", which=DET, width=1000, crop=None):
    f = load_frame(real, ridx[(SCENE, view)])
    dets = runs[which].get((SCENE, view), [])
    gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
    dets, gt = select(dets, gt, show)
    img = render_2d(f["rgb"], f["K"], dets, model, gt, labels="all",
                    title=f"view {view} ({'detector' if which == DET else 'GT masks'}): {show}")
    img = img.crop(crop) if crop else img
    display(img.resize((width, int(width * img.height / img.width))))

views = list(dict.fromkeys([real.frames[0].view, int(pf.confident.idxmin()), int(pf.detections.idxmax())]))
for v in views:
    overlay(v)
overlay(real.frames[0].view, show="confident")            # dense piles: confident poses only are easier to check
if HAS_GT:
    overlay(int(pf.confident.idxmin()), which=GTM)

**대화형 3D** (드래그로 회전, 스크롤로 확대, 박스에 마우스를 올리면 fitness, fit 오차 표시). confident가 가장 많은 장입니다.

In [ ]:
v = int(pf.confident.idxmax())
f = load_frame(real, ridx[(SCENE, v)])
dets = runs[DET].get((SCENE, v), [])
gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
plotly_scene(f["rgb"], f["xyz"], f["valid"], dets, model, gt, max_points=40000,
             title=f"view {v}: detections over the real point cloud, camera frame (mm)").show()

## 6. 판정

위 기준들을 한 표로 모읍니다. **OK**는 기준값 안, **주의**는 결과를 그대로 믿기 전에 확인이 필요함, **-**는 이 데이터로 잴 수 없음입니다.
기준값은 맨 위 `CRIT`에 있고, 근거는 §3-2 보정 표입니다.

In [ ]:
def verdict(ok):
    return "-" if ok is None else ("OK" if ok else "주의")

c = free[free.confident]
n_conf = len(c)
ref_conf = f"참고: 기준 실제 촬영 {ref_free.confident.sum() / len(ref):.1f} / 5" if ref_free is not None else "-"
rows = [
    ("카메라 설정이 학습과 같음 (해상도, K)", f"{this_cam['width']}x{this_cam['height']}, fx {this_cam['fx']:.1f}",
     f"{TRAIN_CAM['width']}x{TRAIN_CAM['height']}, fx {TRAIN_CAM['fx']:.1f}", verdict(CAM_OK)),
    ("렌즈 왜곡 보정", {True: "yes", False: "no", None: "unknown"}[UNDISTORTED], "yes (camera_convert_data.py)",
     verdict(None if UNDISTORTED is None else UNDISTORTED)),
    ("confident pose 수 (장당)", f"{n_conf / len(real):.1f}" + (f" / {PARTS_PER_FRAME} placed" if PARTS_PER_FRAME else ""),
     ">= 80% of placed parts" if PARTS_PER_FRAME else ref_conf,
     verdict(n_conf / (PARTS_PER_FRAME * len(real)) >= 0.8 if PARTS_PER_FRAME else (None if n_conf else False))),
    ("카메라-부품 거리 (confident 중앙값)", f"{c.distance_mm.median():.0f} mm" if n_conf else "-",
     f"학습 {TRAIN_DIST:.0f} mm ± {CRIT['dist_rel']:.0%}",
     verdict(abs(c.distance_mm.median() / TRAIN_DIST - 1) <= CRIT["dist_rel"] if n_conf else None)),
    ("confident 맞춤 오차 중앙값", f"{c.icp_rmse.median():.2f} mm" if n_conf else "-", f"<= {CRIT['fit_rmse_mm']} mm",
     verdict(c.icp_rmse.median() <= CRIT["fit_rmse_mm"] if n_conf else None)),
    ("confident fitness 중앙값", f"{c.icp_fitness.median():.3f}" if n_conf else "-", f">= {CRIT['fitness']}",
     verdict(c.icp_fitness.median() >= CRIT["fitness"] if n_conf else None)),
    ("confident 중 depth 모순", f"{(c.surf_violation > CRIT['violation']).sum()} / {n_conf}" if n_conf else "-",
     f"<= {CRIT['violation_share']:.0%}",
     verdict((c.surf_violation > CRIT["violation"]).mean() <= CRIT["violation_share"] if n_conf else None)),
]
if len(rep):
    rows.append(("반복 촬영 위치 흔들림 (부품별 최대의 중앙값)", f"{rep['centre spread max (mm)'].median():.2f} mm",
                 f"<= {CRIT['repeat_centre_mm']} mm", verdict(rep["centre spread max (mm)"].median() <= CRIT["repeat_centre_mm"])))
    rows.append(("반복 촬영 기울기 흔들림 (부품별 최대의 중앙값)", f"{rep['tilt max (deg)'].median():.2f} deg",
                 f"<= {CRIT['repeat_tilt_deg']} deg", verdict(rep["tilt max (deg)"].median() <= CRIT["repeat_tilt_deg"])))
else:
    rows.append(("반복 촬영 흔들림", "-", "같은 장면 2장 이상 필요", "-"))
if HAS_GT:
    m = tables[DET][tables[DET].matched & tables[DET].confident]
    rows.append(("confident가 의사 정답과 1mm 이내", f"{(m.final_add_s < 1).mean():.1%}" if len(m) else "-", ">= 95%",
                 verdict((m.final_add_s < 1).mean() >= 0.95 if len(m) else None)))
verdict_df = pd.DataFrame(rows, columns=["기준", "이 데이터", "기준값", "판정"]).set_index("기준")
display(verdict_df)
bad = verdict_df[verdict_df["판정"] == "주의"].index.tolist()
print("종합: " + ("측정 가능한 모든 기준을 통과했습니다. confident pose는 depth와 모순 없이 맞고, 수치가 기준 실제 촬영(의사 정답과 1mm 이내)과 같은 수준입니다 (GT 없는 판정의 한계는 아래 주의점 참고)."
                 if not bad else "확인 필요 -> " + ", ".join(bad)))

**주의점**
- **GT 없는 판정은 일관성 검사입니다.** depth와 모순이 없고 반복해서 같은 값이 나온다는 뜻이지, 정답과 같다는 증명은 아닙니다.
  특히 depth로 확인되지 않는 방향(옆 방향 위치, 허브 축 둘레 각도)은 틀려도 이 기준들을 통과할 수 있습니다.
  최종 확인은 로봇 집기 성공률 같은 독립적인 기준, 또는 몇 장에 대한 의사 정답(FoundationPose 등)으로 하세요.
- **카메라 설정이 다르면(§1) 다른 숫자는 의미가 없습니다.** 먼저 학습과 같은 설정(1236×1032, PrimaryCamera, 컬러 텍스처)으로 다시 찍으세요.
- **렌즈 왜곡:** 파이프라인은 왜곡 없는 pinhole K로 depth를 3D로 바꿉니다. 왜곡 보정 없이 넣으면 이미지 가장자리로 갈수록 수 mm의 옆 방향 오차가 생기고,
  이 오차는 위 기준들로는 보이지 않습니다. `data/pseudo_gt_20_bop`(기준)은 왜곡 보정 여부를 알 수 없습니다.
- **표본:** 장 수와 부품 수가 적으면 비율 하나가 크게 흔들립니다. 배치를 바꿔 10장 이상, 가능하면 시뮬레이션처럼 가득 찬 bin도 찍어 보세요.

**다음 단계 (판정이 '주의'일 때)**
1. **카메라 설정 / 왜곡 보정 / 거리**가 '주의'면 촬영부터 다시 합니다. 나머지 기준은 그 뒤에 봅니다.
2. **confident 수가 적고 §4의 anchor dz가 크면** 네트워크 깊이 오차입니다. depth anchor 설정(`refine.depth_anchor*`)과 ICP 허용 범위를 확인하고,
   부족하면 시뮬레이션 데이터를 실제 조건(배치, depth 노이즈와 결측)에 맞춰 다시 학습합니다.
3. **confident 중 depth 모순이 있으면** 그 검출을 §5 그림에서 확인합니다. 오검출이 confident로 표시된 것이면 confidence 기준을 조입니다.
4. **반복 흔들림이 크면** 촬영 노이즈(노출, 반사)나 ICP 수렴 문제입니다. 같은 장면을 더 찍어 흔들림이 어느 부품에서 나오는지 봅니다.